# OnlinePreprocessor — Manual Validation Notebook

Validates `OnlinePreprocessor.process_batch()` interactively.

Can run in two modes:
- **Synthetic** (default) — no real data needed; uses random matrices.
- **Real** — loads a saved `decoder_pipeline.joblib` and uses the exported `online_state`.

**This notebook is for validation only — not part of the app.**

In [ ]:
import sys
import time
from pathlib import Path

# Walk up the directory tree to find the src/ directory robustly,
# regardless of where Jupyter was launched from.
_search = Path().resolve()
while _search != _search.parent:
    if (_search / "src" / "backend").exists():
        if str(_search / "src") not in sys.path:
            sys.path.insert(0, str(_search / "src"))
        break
    _search = _search.parent

import matplotlib
matplotlib.use("inline")
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import welch

from backend.online_phase import OnlinePreprocessor

print("Imports OK")

---
## Configuration

Set `USE_REAL_ARTIFACT = True` and point `ARTIFACT_PATH` at a saved `decoder_pipeline.joblib`
to validate against real ICA matrices.
Leave `False` to run fully synthetically.

In [ ]:
# ── Toggle between synthetic and real modes ───────────────────────────────
USE_REAL_ARTIFACT = False
ARTIFACT_PATH = Path("../../data_vault/decoder_pipeline.joblib")  # only used if USE_REAL_ARTIFACT=True

INPUT_SFREQ  = 1000.0   # Hz — LSL stream rate
TARGET_SFREQ = 256      # Hz — must match artifact / synthetic settings
BATCH_SIZE   = 40       # samples — 40 ms at 1000 Hz
N_CHANNELS   = 20       # channels (synthetic mode only; real mode reads from artifact)
N_COMPONENTS = 4        # ICA components (synthetic mode only)
# ─────────────────────────────────────────────────────────────────────────
print(f"Mode: {'real artifact' if USE_REAL_ARTIFACT else 'synthetic'}")

---
## Build OnlinePreprocessor

In [ ]:
if USE_REAL_ARTIFACT:
    from backend.online_phase import load_decoder_pipeline_artifact
    from backend.core.settings_manager import SettingsManager

    artifact = load_decoder_pipeline_artifact(ARTIFACT_PATH)
    settings = SettingsManager(Path("../experiment_config.yaml"))
    preprocessing_settings = settings.preprocessing.model_dump()
    online_state = artifact.online_state
    n_channels = len(online_state["ch_names"])
else:
    rng = np.random.default_rng(0)
    pca = rng.standard_normal((N_COMPONENTS, N_CHANNELS))
    unmix = rng.standard_normal((N_COMPONENTS, N_COMPONENTS))
    online_state = {
        "bad_channels": [],
        "interp_weights": None,
        "ch_names": [f"CH{i:03d}" for i in range(N_CHANNELS)],
        "ica_unmixing": unmix,
        "ica_mixing": np.linalg.pinv(unmix),
        "ica_pca_components": pca,
        "ica_pca_mean": np.zeros(N_CHANNELS),
        "ica_exclude": [0],
        "pre_whitener": np.ones((N_CHANNELS, 1)),
        "sfreq_offline": float(TARGET_SFREQ),
    }
    preprocessing_settings = {
        "bandpass": {"l_freq": 1.0, "h_freq": 40.0, "method": "iir", "notch": 50.0},
        "resample": {"target_rate": TARGET_SFREQ},
    }
    n_channels = N_CHANNELS

preprocessor = OnlinePreprocessor(
    preprocessing_settings=preprocessing_settings,
    online_state=online_state,
    input_sfreq=INPUT_SFREQ,
)

budget_ms = BATCH_SIZE / INPUT_SFREQ * 1000
print(f"n_channels   : {preprocessor.n_channels}")
print(f"input_sfreq  : {preprocessor.input_sfreq} Hz")
print(f"target_sfreq : {preprocessor.target_sfreq} Hz")
print(f"batch budget : {budget_ms:.1f} ms")
print(f"bad channels : {online_state['bad_channels']}")
print(f"ICA exclude  : {online_state['ica_exclude']}")

# ── Assertions ────────────────────────────────────────────────────────────
assert preprocessor.n_channels == n_channels
assert preprocessor.input_sfreq == INPUT_SFREQ
assert preprocessor.target_sfreq == TARGET_SFREQ
print("\n✓ Constructor OK")

---
## Single Batch — Shape and Types

In [ ]:
rng = np.random.default_rng(1)
batch = rng.standard_normal((BATCH_SIZE, n_channels)) * 1e-5
timestamps = np.arange(BATCH_SIZE, dtype=float) / INPUT_SFREQ

features, out_ts = preprocessor.process_batch(batch, timestamps)

expected_n_out = int(BATCH_SIZE * TARGET_SFREQ / INPUT_SFREQ)
print(f"Input  : {batch.shape}  @ {INPUT_SFREQ:.0f} Hz")
print(f"Output : {features.shape}  @ {TARGET_SFREQ} Hz")
print(f"Timestamps: {out_ts.shape}  range [{out_ts[0]:.4f}, {out_ts[-1]:.4f}] s")
print(f"Expected ~{expected_n_out} output samples")

# ── Assertions ────────────────────────────────────────────────────────────
assert features.ndim == 2,              "output must be 2D"
assert features.shape[1] == n_channels, "channel count must be preserved"
assert out_ts.shape[0] == features.shape[0], "timestamps must match rows"
assert features.dtype == float,         "output dtype must be float"
assert abs(features.shape[0] - expected_n_out) <= 1, (
    f"Expected ~{expected_n_out} output samples, got {features.shape[0]}"
)
print("\n✓ Single batch OK")

---
## Streaming Simulation — 10 Sequential Batches

Simulates LSLReceiver delivering 40-sample chunks continuously.
Validates that state carries forward correctly (chunk total == one-shot total).

In [ ]:
N_BATCHES = 10
n_total = BATCH_SIZE * N_BATCHES

rng = np.random.default_rng(2)
full_data = rng.standard_normal((n_total, n_channels)) * 1e-5
full_ts   = np.arange(n_total, dtype=float) / INPUT_SFREQ

# ── One-shot reference ────────────────────────────────────────────────────
preprocessor.reset_state()
ref_out, ref_ts = preprocessor.process_batch(full_data, full_ts)

# ── Chunked streaming ─────────────────────────────────────────────────────
preprocessor.reset_state()
chunk_outs, chunk_ts = [], []
for i in range(N_BATCHES):
    s = i * BATCH_SIZE
    o, t = preprocessor.process_batch(full_data[s:s+BATCH_SIZE], full_ts[s:s+BATCH_SIZE])
    chunk_outs.append(o)
    chunk_ts.append(t)
    print(f"  batch {i+1:2d}: {o.shape[0]:3d} output samples")

chunked_out = np.concatenate(chunk_outs)
chunked_ts  = np.concatenate(chunk_ts)

print(f"\nOne-shot  : {ref_out.shape[0]} samples")
print(f"Chunked   : {chunked_out.shape[0]} samples")

# ── Assertions ────────────────────────────────────────────────────────────
assert ref_out.shape[0] == chunked_out.shape[0], "sample counts must match"
np.testing.assert_allclose(ref_out, chunked_out, atol=1e-10,
    err_msg="chunked output must be VALUES-identical to one-shot")
np.testing.assert_allclose(ref_ts, chunked_ts, atol=1e-12)
print("\n✓ Streaming simulation OK (chunked == one-shot values)")

---
## Frequency Content — Bandpass Verification

Process a long synthetic signal and check that the bandpass filter is applied:
power above h_freq and below l_freq must be strongly attenuated.

In [ ]:
n_sec = 10
n_samples = int(INPUT_SFREQ * n_sec)

# Synthetic broadband signal
rng = np.random.default_rng(3)
signal = rng.standard_normal((n_samples, n_channels)) * 1e-5
ts     = np.arange(n_samples, dtype=float) / INPUT_SFREQ

preprocessor.reset_state()
out, out_ts = preprocessor.process_batch(signal, ts)

# PSD of first channel at output rate
f, pxx = welch(out[:, 0], fs=TARGET_SFREQ, nperseg=min(256, len(out)))

l_freq = preprocessing_settings["bandpass"]["l_freq"]
h_freq = preprocessing_settings["bandpass"]["h_freq"]

# Power in passband vs. stopband
# highstop starts 20 Hz above the cutoff to stay clear of the IIR transition band
passband  = pxx[(f >= l_freq + 2) & (f <= h_freq - 2)]
highstop  = pxx[f > h_freq + 20]
pwr_ratio_db = 10 * np.log10(passband.mean() / (highstop.mean() + 1e-30))
print(f"Passband mean power  : {10*np.log10(passband.mean()+1e-30):.1f} dB")
print(f"High-stop mean power : {10*np.log10(highstop.mean()+1e-30):.1f} dB")
print(f"Passband/stopband    : {pwr_ratio_db:.1f} dB (expect > 30 dB)")

plt.figure(figsize=(10, 3))
plt.semilogy(f, pxx, label="ch0 output")
plt.axvspan(0, l_freq, alpha=0.15, color="red", label="stopband")
plt.axvspan(h_freq, TARGET_SFREQ / 2, alpha=0.15, color="red")
plt.xlabel("Frequency (Hz)")
plt.ylabel("PSD")
plt.title(f"Output PSD — bandpass {l_freq}–{h_freq} Hz")
plt.legend()
plt.tight_layout()
plt.show()

# ── Assertion ─────────────────────────────────────────────────────────────
assert pwr_ratio_db > 30, f"Bandpass attenuation too low: {pwr_ratio_db:.1f} dB"
print("\n✓ Frequency content OK")

---
## Average Reference — Zero Mean Check

In [ ]:
# Average reference is applied inside process_batch() before ICA.
# ICA (step 5) applies a linear channel mix that perturbs the row mean,
# so the final output is NOT expected to have zero mean across channels.
row_means = out.mean(axis=1)
max_mean = np.abs(row_means).max()
print(f"Max |row mean| after full pipeline: {max_mean:.2e}")
print("(Non-zero is expected — ICA runs after avg ref and perturbs channel mean)")

plt.figure(figsize=(10, 2))
plt.plot(out_ts, row_means)
plt.axhline(0, color="k", linewidth=0.5)
plt.xlabel("Time (s)")
plt.ylabel("Mean across channels")
plt.title("Per-sample channel mean after full pipeline (avg ref → ICA)")
plt.tight_layout()
plt.show()
print("✓ Average reference cell OK (no assertion — ICA perturbs mean)")

---
## Latency Benchmark

Measures per-batch wall-clock time over many iterations.
The 40-sample budget at 1000 Hz is **40 ms**.

In [ ]:
N_BENCH  = 2000
N_WARMUP = 50

rng = np.random.default_rng(99)
preprocessor.reset_state()
times_ms = []

for i in range(N_WARMUP + N_BENCH):
    b  = rng.standard_normal((BATCH_SIZE, n_channels)) * 1e-5
    ts = np.arange(BATCH_SIZE, dtype=float) / INPUT_SFREQ + i * BATCH_SIZE / INPUT_SFREQ
    t0 = time.perf_counter()
    preprocessor.process_batch(b, ts)
    t1 = time.perf_counter()
    if i >= N_WARMUP:
        times_ms.append((t1 - t0) * 1000)

arr = np.array(times_ms)
budget_ms = BATCH_SIZE / INPUT_SFREQ * 1000

print(f"=== Latency (ms)  —  {N_BENCH} batches ===")
print(f"  mean  {arr.mean():.3f}   std  {arr.std():.3f}")
print(f"  p50   {np.percentile(arr, 50):.3f}")
print(f"  p95   {np.percentile(arr, 95):.3f}")
print(f"  p99   {np.percentile(arr, 99):.3f}")
print(f"  max   {arr.max():.3f}")
print(f"\n=== Budget usage (budget = {budget_ms:.1f} ms) ===")
print(f"  mean  {100*arr.mean()/budget_ms:.1f} %")
print(f"  p99   {100*np.percentile(arr, 99)/budget_ms:.1f} %")

plt.figure(figsize=(10, 3))
plt.hist(arr, bins=60, edgecolor="none")
plt.axvline(np.percentile(arr, 99), color="red", linestyle="--", label="p99")
plt.axvline(budget_ms, color="orange", linestyle="--", label=f"budget ({budget_ms:.0f} ms)")
plt.xlabel("process_batch() wall time (ms)")
plt.ylabel("Count")
plt.title("Per-batch latency distribution")
plt.legend()
plt.tight_layout()
plt.show()

---
## Summary

If all cells passed:
- `OnlinePreprocessor` constructs correctly from both synthetic and real offline state.
- Output shape, dtype, and timestamps are correct.
- Chunked streaming produces values identical to single-pass processing.
- Bandpass filter is applied (high-frequency power attenuated > 20 dB).
- Average reference makes per-sample channel mean ≈ 0.
- Latency is well within the 40 ms real-time budget.

**Next step:** wire `OnlinePreprocessor` into `StreamWorker` → `LiveInferenceEngine`.